In [ ]:
import os
import glob
import requests
import time
import logging
import gzip
import pandas as pd
from typing import Set
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [ ]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


In [ ]:
import os
import requests
import time
import logging
import gzip
import pandas as pd
from typing import Set
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class AlphaFoldTracker:
    def __init__(self):
        self.successful_downloads = 0
        self.already_exists = 0
        self.api_404_not_modeled = []
        self.other_errors = []

    def print_summary(self):
        print("\n" + "="*50)
        print("🧬 ALPHAFOLD DOWNLOAD SUMMARY")
        print("="*50)
        print(f"Successfully Downloaded:    {self.successful_downloads}")
        print(f"Already Existed Locally:    {self.already_exists}")
        print(f"Unmodeled Proteins (404):   {len(self.api_404_not_modeled)}")
        print(f"Other Errors:               {len(self.other_errors)}")
        print("="*50 + "\n")

class AlphaFoldDownloader:
    def __init__(self, output_dir: str = "alphafold_cif", email: str = ""):
        self.output_dir = output_dir
        self.tracker = AlphaFoldTracker()
        
        if not os.path.exists(self.output_dir):
            os.makedirs(self.output_dir)

        self.session = requests.Session()
        
        # Transparent, compliant headers as requested by EBI's API guidelines
        self.session.headers.update({
            # 'User-Agent': f'ProteinLigandDatasetCurator/1.0 (mailto:{email})',
            'User-Agent': f'ProteinLigandDatasetCurator/1.0',
            
            'Accept-Encoding': 'gzip, deflate'
        })
        
        retries = Retry(total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
        self.session.mount('https://', HTTPAdapter(max_retries=retries))

    def download_structures(self, uniprot_ids: Set[str]) -> Set[str]:
        total = len(uniprot_ids)
        logger.info(f"Starting compliant download for {total} unique UniProt IDs...")
        valid_uniprot_ids = set()

        for i, raw_uid in enumerate(uniprot_ids):
            uid = str(raw_uid).strip()
            if not uid:
                continue

            if i > 0 and i % 100 == 0:
                logger.info(f"Progress: {i}/{total} structures processed...")

            file_path = os.path.join(self.output_dir, f"AF-{uid}.cif.gz")
            
            if os.path.exists(file_path):
                self.tracker.already_exists += 1
                valid_uniprot_ids.add(uid)
                continue

            # Step 1: Query the official API for the exact file location
            api_url = f"https://alphafold.ebi.ac.uk/api/prediction/{uid}"

            try:
                api_response = self.session.get(api_url, timeout=10)
                time.sleep(0.1) # Be polite to EBI's API
                
                if api_response.status_code == 404:
                    self.tracker.api_404_not_modeled.append(uid)
                    continue
                
                api_response.raise_for_status()
                data = api_response.json()
                
                if not data or not isinstance(data, list):
                    self.tracker.other_errors.append((uid, "Invalid JSON structure"))
                    continue

                # Get the verified CIF URL from the payload
                cif_url = data[0].get("cifUrl")
                if not cif_url:
                    self.tracker.other_errors.append((uid, "Missing cifUrl in payload"))
                    continue

                # Step 2: Download the structure from the exact URL
                with self.session.get(cif_url, stream=True, timeout=15) as cif_response:
                    time.sleep(0.1) # Be polite to the Google Cloud bucket hosting the file
                    
                    if cif_response.status_code == 200:
                        with gzip.open(file_path, 'wb') as f:
                            for chunk in cif_response.iter_content(chunk_size=8192):
                                if chunk:
                                    f.write(chunk)
                        
                        self.tracker.successful_downloads += 1
                        valid_uniprot_ids.add(uid)
                    else:
                        self.tracker.other_errors.append((uid, f"CIF Download Error: {cif_response.status_code}"))
                    
            except requests.exceptions.RequestException as e:
                self.tracker.other_errors.append((uid, str(e)))

        self.tracker.print_summary()
        return valid_uniprot_ids

def run_phase_4(df: pd.DataFrame) -> pd.DataFrame:
    unique_uniprot = set(df['UniProt_ID'].dropna().astype(str).str.strip().unique())
    
    # Initialize with a compliant email for the User-Agent
    downloader = AlphaFoldDownloader(output_dir="alphafold_cif", email="")
    valid_uniprot_ids = downloader.download_structures(unique_uniprot)
    
    df['UniProt_ID'] = df['UniProt_ID'].astype(str).str.strip()
    df_filtered = df[df['UniProt_ID'].isin(valid_uniprot_ids)].copy()
    
    return df_filtered

In [ ]:
df = pd.read_csv("step_2_with_smiles.csv", index_col = 0)

In [ ]:
lst = glob.glob("alphafold_cif/*")

In [ ]:
proteins_cached = []
for x in lst:
    proteins_cached.append(x.split("/")[-1].split("AF-")[1].split(".")[0])

In [ ]:
df_ = df[~df["UniProt_ID"].isin(proteins_cached)]

In [ ]:
df["UniProt_ID"].nunique(), df_["UniProt_ID"].nunique()

In [ ]:
filtered_df = run_phase_4(df_)

In [ ]:
filtered_df.head(10)

In [ ]:
import requests

uniprot_id = "Q9M040"
api_url = f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot_id}"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

# 1. Ask the API for the exact file location
api_response = requests.get(api_url, headers=headers)

if api_response.status_code == 200:
    data = api_response.json()
    # EBI returns a list of fragments; we'll grab the first one
    if len(data) > 0:
        actual_cif_url = data[0].get("cifUrl")
        print(f"API returned exact URL: {actual_cif_url}")
        
        # 2. Now download the actual file
        cif_response = requests.get(actual_cif_url, headers=headers)
        print(f"Download Status: {cif_response.status_code}, Bytes: {len(cif_response.content)}")
    else:
        print("API returned an empty list (protein not modeled).")
else:
    print(f"API failed with status: {api_response.status_code}")

In [ ]:
print(cif_response.text)

In [ ]:
df.head(1)

In [ ]:
from tqdm import tqdm

In [ ]:
for i in tqdm(range(0, 100, 2)):
    continue